Fraud Detection: Exploratory Data Analysis

This notebook explores the fraud detection dataset to understand the distribution of transaction types, urgency levels, and financial patterns. The goal is to surface insights that inform feature engineering and model design.

Dataset: Simulated payment transactions (PaySim)  
Target: `urgency_level`: 0 (No Action) to 3 (Immediate Action)

Importing libraries, setting up plots and reading in the data

In [ ]:
# importing libraries 
import pandas as pd               # dataframes and CSV loading
import numpy as np                # numerical operations
import matplotlib.pyplot as plt   # base plotting library
import matplotlib.ticker as mticker  # custom axis tick formatting (e.g. 1,000 instead of 1000)
import matplotlib
import seaborn as sns             # higher-level plotting built on matplotlib


# darkgrid adds a grey background with gridlines, 'muted' is a softer colour palette
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120          # higher DPI = sharper plots
plt.rcParams['figure.figsize'] = (10, 5)  # default figure size in inches

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = True

# Human-readable labels for each urgency level, used in chart titles and legends
URGENCY_LABELS = {
    0: '0 - No Action',
    1: '1 - Monitor',
    2: '2 - Review',
    3: '3 - Immediate Action'
}

# Colour scale: green (safe) to yellow to orange to red (urgent)
# These map to urgency levels 0, 1, 2, 3 respectively
URGENCY_COLORS = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']

# load data
df = pd.read_csv("C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/data/train.csv")

# Strip any accidental leading/trailing whitespace from column names
# (common issue when CSVs are exported from Excel)
df.columns = df.columns.str.strip()

# Ensure the 'type' column is treated as a string (categorical), not a generic object
df['type'] = df['type'].astype(str)

# Add a human-readable label column for use in plots
# .map() replaces each integer value with its corresponding string from URGENCY_LABELS
df['urgency_label'] = df['urgency_level'].map(URGENCY_LABELS)

print(f'Dataset shape: {df.shape}')  # (rows, columns)
df.head()                            # preview the first 5 rows

Class Distribution (Urgency Level Counts)

The first thing to check in any classification problem is how balanced the target variable is.
If one class dominates, a model could score high accuracy simply by always predicting that class, which would be useless in practice.
Understanding the imbalance here tells us what evaluation metric to use and whether we need to correct for it during training.

In [ ]:
# Count how many transactions belong to each urgency level
# .sort_index() ensures they appear in order 0, 1, 2, 3
counts = df['urgency_level'].value_counts().sort_index()

# Calculate what percentage of the total each class represents
# Dividing by .sum() gives proportions; multiplying by 100 converts to percentages
pcts = (counts / counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(9, 5))

# Bar chart 
bars = ax.bar(
    [URGENCY_LABELS[i] for i in counts.index],  # x-axis: label names instead of 0/1/2/3
    counts.values,                               # y-axis: transaction counts
    color=URGENCY_COLORS,                        # colour each bar by urgency level
    edgecolor='white',                           # white outline between bars
    linewidth=0.8
)
ax.set_title('Transaction Count by Urgency Level', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Transactions')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)  # tilt x-axis labels so they don't overlap

# Format y-axis numbers with commas (e.g. 10,000 instead of 10000)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Add the exact count as a label on top of each bar
# Also adds the percentage in brackets so both pieces of info are visible at once
for bar, count, pct in zip(bars, counts.values, pcts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # x position: centre of bar
        bar.get_height() + 50,              # y position: just above the bar
        f'{count:,}\n({pct}%)',             # count on first line, percentage on second
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()  # automatically adjusts spacing so nothing gets clipped
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_class_distribution.png', bbox_inches='tight')  # save to outputs folder
plt.show()

# Print a clean summary table
print('\nClass breakdown:')
summary = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
print(summary.to_string())

Transaction Type Breakdowns

There are 5 transaction types: `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER`.
Not all types are equally risky, for example, a `TRANSFER` moving money out of an account is inherently more suspicious than a `CASH_IN`.
This section looks at both the volume of each type and the fraud concentration within each type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# total transaction count per type 
# .value_counts() returns counts sorted from most to least frequent
type_counts = df['type'].value_counts()

axes[0].bar(
    type_counts.index,   # transaction type names on x-axis
    type_counts.values,  # counts on y-axis
    color=sns.color_palette('muted', len(type_counts)),  # unique colour per type
    edgecolor='white'
)
axes[0].set_title('Transaction Volume by Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Label each bar with the exact count
for i, (idx, val) in enumerate(type_counts.items()):
    axes[0].text(i, val + 50, f'{val:,}', ha='center', va='bottom', fontsize=9)

# urgency breakdown per type as stacked percentage bar 
# groupby(['type', 'urgency_level']).size() counts how many rows have each combination
# .unstack() pivots urgency_level into columns so each row is a transaction type
# fill_value=0 fills missing combinations with 0 (e.g. if a type has no class 3 transactions)
type_urgency = df.groupby(['type', 'urgency_level']).size().unstack(fill_value=0)

# Convert raw counts to percentages so all bars reach 100%
# .div(..., axis=0) divides each row by that row's total count
type_urgency_pct = type_urgency.div(type_urgency.sum(axis=1), axis=0) * 100

type_urgency_pct.plot(
    kind='bar', stacked=True, ax=axes[1],
    color=URGENCY_COLORS,
    edgecolor='white', linewidth=0.5
)
axes[1].set_title('Urgency Level Breakdown by Transaction Type (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Transaction Type')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(
    [URGENCY_LABELS[i] for i in sorted(df['urgency_level'].unique())],
    title='Urgency Level',
    bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8  # place legend outside the plot
)

plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_transaction_types.png', bbox_inches='tight')
plt.show()

# Fraud rate per type 
# For each transaction type, calculate the % of transactions with urgency > 0
# lambda x applies a custom function to each group:
#   (x['urgency_level'] > 0).sum() = count of flagged transactions in this group
#   / len(x) = divide by total transactions in this group
#   * 100 = convert to percentage
print('\nFraud rate (urgency > 0) by transaction type:')
fraud_rate = df.groupby('type').apply(
    lambda x: (x['urgency_level'] > 0).sum() / len(x) * 100
).round(2).sort_values(ascending=False)
print(fraud_rate.to_string())

Amount Distributions by Urgency Level

Intuitively, fraudsters may attempt to move large sums of money quickly. This section tests whether transaction amount is actually higher for flagged transactions, and whether the relationship is consistent across all urgency levels. We also look at the shape of the distribution, is it normally distributed or skewed?

In [ ]:
# Create a 2x2 grid of subplots: one histogram per urgency level
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()  # flatten from 2D array to 1D list so we can loop over it easily

for i, level in enumerate([0, 1, 2, 3]):
    # Filter to only transactions with this urgency level
    subset = df[df['urgency_level'] == level]['amount']

    # Cap at the 99th percentile to prevent a small number of extreme outliers
    # from squashing the rest of the histogram into a thin line on the left
    cap = subset.quantile(0.99)
    subset_capped = subset[subset <= cap]

    # Draw histogram with 60 bins
    axes[i].hist(
        subset_capped,
        bins=60,
        color=URGENCY_COLORS[i],
        edgecolor='white', linewidth=0.4,
        alpha=0.85  # slight transparency so gridlines show through
    )
    axes[i].set_title(f'{URGENCY_LABELS[level]}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Transaction Amount')
    axes[i].set_ylabel('Count')

    # Format x-axis with commas for readability
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    # Draw vertical lines for mean and median
    # Mean is pulled toward outliers; median is more robust, comparing them shows skewness
    axes[i].axvline(subset.mean(), color='black', linestyle='--', linewidth=1.2,
                    label=f'Mean: {subset.mean():,.0f}')
    axes[i].axvline(subset.median(), color='grey', linestyle=':', linewidth=1.2,
                    label=f'Median: {subset.median():,.0f}')
    axes[i].legend(fontsize=8)

plt.suptitle(
    'Transaction Amount Distribution by Urgency Level\n(capped at 99th percentile)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_amount_distributions.png', bbox_inches='tight')
plt.show()

# Print full descriptive statistics (count, mean, std, min, quartiles, max)
print('\nAmount statistics by urgency level:')
print(df.groupby('urgency_level')['amount'].describe().round(2).to_string())

In [ ]:
# Box plot: compare all urgency levels side by side 
# Box plots show median, IQR (25th-75th percentile), and outliers in one view
# This makes it easy to see if the median amount increases with urgency level

fig, ax = plt.subplots(figsize=(11, 5))

# Build a list of 4 arrays, one per urgency level
# .clip(upper=...) removes extreme outliers so the boxes are visible
data_by_level = [
    df[df['urgency_level'] == i]['amount'].clip(upper=df['amount'].quantile(0.99))
    for i in range(4)
]

# patch_artist=True fills the box with colour; notch=True adds a notch around the median
# A notched box plot gives a rough 95% confidence interval around the median
bp = ax.boxplot(
    data_by_level,
    patch_artist=True,
    notch=True,
    medianprops=dict(color='white', linewidth=2)  # white median line so it stands out
)

# Apply the urgency colour to each box
for patch, color in zip(bp['boxes'], URGENCY_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_xticklabels([URGENCY_LABELS[i] for i in range(4)])
ax.set_title('Transaction Amount by Urgency Level (Box Plot)', fontsize=13, fontweight='bold')
ax.set_ylabel('Transaction Amount')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_amount_boxplot.png', bbox_inches='tight')
plt.show()

Correlation Heatmap

A correlation heatmap shows the linear relationship between every pair of numerical features. Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation), with 0 meaning no linear relationship.

We include both the raw features and the engineered features here to see which ones are most predictive of `urgency_level`. We also check for multicollinearity, when two features are highly correlated with each other, which can cause redundancy.

In [ ]:
# Work on a copy so we don't modify the original dataframe
df_corr = df.copy()

ATOL = 1e-6  # tolerance for float comparisons (avoids false positives from rounding errors)

# Compute the 99.9th percentile amount threshold from training data
# Using training data only (not test) prevents data leakage
AMOUNT_THRESHOLD = df_corr['amount'].quantile(0.999)

# Engineered features 
# deltaOrg: how much the origin account balance changed (positive = money left the account)
df_corr['deltaOrg']  = df_corr['oldbalanceOrg']  - df_corr['newbalanceOrig']

# deltaDest: how much the destination account balance changed (positive = money arrived)
df_corr['deltaDest'] = df_corr['newbalanceDest'] - df_corr['oldbalanceDest']

# Binary flag: 1 if the transaction amount is unusually large (top 0.1%)
df_corr['check_Amount_Size'] = (df_corr['amount'] > AMOUNT_THRESHOLD).astype(int)

# Binary flag: 1 if the destination account starts with 'M' (merchant accounts)
df_corr['dest_is_merchant']  = df_corr['nameDest'].str.startswith('M').astype(int)

# Encode the 'type' column as integers so it can appear in the correlation matrix
# pd.Categorical assigns an integer code to each unique string value
df_corr['type_encoded'] = pd.Categorical(df_corr['type']).codes

# Select only numeric columns for the correlation matrix
numeric_cols = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'deltaOrg', 'deltaDest',
    'check_Amount_Size', 'dest_is_merchant',
    'type_encoded', 'step', 'urgency_level'
]

# .corr() computes Pearson correlation coefficients between all column pairs
corr_matrix = df_corr[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))

# Create a mask to hide the upper triangle (it's a mirror of the lower triangle)
# np.triu returns True for the upper triangle; masking these cells keeps the plot cleaner
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,        # show the correlation value inside each cell
    fmt='.2f',         # format to 2 decimal places
    cmap='RdYlGn',     # red = negative, yellow = neutral, green = positive
    center=0,          # centre the colour scale at 0
    vmin=-1, vmax=1,   # fix scale from -1 to 1
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 8},
    ax=ax
)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Print the features most correlated with our target, sorted by absolute value
# abs() sorts by strength of correlation regardless of direction
print('\nTop correlations with urgency_level:')
target_corr = corr_matrix['urgency_level'].drop('urgency_level').sort_values(
    key=abs, ascending=False
)
print(target_corr.round(4).to_string())

Balance Discrepancy Analysis

For each transaction type there is an expected relationship between the old balance, the amount, and the new balance. For example, a `CASH_OUT` should decrease the origin account by exactly `amount`. If the balance does not update as expected, this is a red flag, and it may indicate manipulation, a system error, or fraudulent activity.

This section checks whether these balance discrepancies are more common in higher-urgency transactions.

In [ ]:
# Work on a copy to avoid modifying the original dataframe
df_bal = df.copy()

# Initialise both flag columns to 0 (no discrepancy)
# We will overwrite these for each transaction type below
df_bal['check_balanceOrg_tol']  = 0
df_bal['check_balanceDest_tol'] = 0

# Define the expected balance update rule for each transaction type:
# 'org_decrease' = origin balance should go DOWN by amount
# 'org_increase' = origin balance should go UP by amount (e.g. receiving cash)
# 'dest_increase' = destination balance should go UP by amount
# None = destination balance is not expected to change (e.g. merchant accounts)
type_rules = {
    'PAYMENT':  ('org_decrease', None),          # sender pays, no dest balance update
    'CASH_IN':  ('org_increase', None),           # account receives cash
    'CASH_OUT': ('org_decrease', None),           # account withdraws cash
    'TRANSFER': ('org_decrease', 'dest_increase'),# money moves from origin to destination
    'DEBIT':    ('org_decrease', 'dest_increase'),# similar to transfer
}

# Loop through each transaction type and apply the appropriate balance check
for tx_type, (org_rule, dest_rule) in type_rules.items():
    # Create a boolean mask for rows of this transaction type
    mask = df_bal['type'] == tx_type

    # Calculate what the new origin balance SHOULD be after this transaction
    if org_rule == 'org_decrease':
        exp_org = df_bal.loc[mask, 'oldbalanceOrg'] - df_bal.loc[mask, 'amount']
    else:  # org_increase
        exp_org = df_bal.loc[mask, 'oldbalanceOrg'] + df_bal.loc[mask, 'amount']

    # np.isclose checks if two floats are approximately equal within the tolerance ATOL
    # We use this instead of == because floating point arithmetic is imprecise
    # The ~ (tilde) negates it: True where they are NOT close (i.e. a discrepancy exists)
    # .astype(int) converts True/False to 1/0
    df_bal.loc[mask, 'check_balanceOrg_tol'] = (
        ~np.isclose(df_bal.loc[mask, 'newbalanceOrig'], exp_org, atol=ATOL)
    ).astype(int)

    # Only check destination balance for types where it should change
    if dest_rule == 'dest_increase':
        exp_dst = df_bal.loc[mask, 'oldbalanceDest'] + df_bal.loc[mask, 'amount']
        df_bal.loc[mask, 'check_balanceDest_tol'] = (
            ~np.isclose(df_bal.loc[mask, 'newbalanceDest'], exp_dst, atol=ATOL)
        ).astype(int)

# Plot discrepancy rates per urgency level 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Loop over both flags and create a bar chart for each
for ax, flag, title in zip(
    axes,
    ['check_balanceOrg_tol', 'check_balanceDest_tol'],
    ['Origin Balance Discrepancy vs Urgency Level',
     'Destination Balance Discrepancy vs Urgency Level']
):
    # Count how many transactions at each urgency level have/don't have a discrepancy
    # .unstack() pivots the flag (0 or 1) into columns
    ct = df_bal.groupby(['urgency_level', flag]).size().unstack(fill_value=0)

    # Convert to percentages so we can compare across urgency levels fairly
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

    ct_pct.plot(
        kind='bar', ax=ax,
        color=['#4CAF50', '#F44336'],  # green = no discrepancy, red = discrepancy
        edgecolor='white', width=0.6
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Urgency Level')
    ax.set_ylabel('Percentage (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(['No Discrepancy (0)', 'Discrepancy Detected (1)'], fontsize=9)

plt.suptitle('Balance Discrepancy Rate by Urgency Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/images/plot_balance_discrepancy.png', bbox_inches='tight')
plt.show()

# Print the exact discrepancy rates as percentages for each urgency level
# .mean() on a binary (0/1) column gives the proportion of 1s
# multiplying by 100 converts to percentage
print('\nOrigin balance discrepancy rate by urgency level (%):')
print(
    df_bal.groupby('urgency_level')['check_balanceOrg_tol']
    .mean().mul(100).round(2).to_string()
)
print('\nDestination balance discrepancy rate by urgency level (%):')
print(
    df_bal.groupby('urgency_level')['check_balanceDest_tol']
    .mean().mul(100).round(2).to_string()
)